# Sample Analysis


In [1]:
from pathlib import Path
import os
import re

import numpy as np
import pandas as pd

from HW_HD_utils import *
from plots_utils import *
from create_npy_file import create_sample_npy_files


In [2]:
sample_registers = ["r0", "r1", "r2", "r3", "r4", "r5", "r6", "r7", "r8", "r9", "r10", "r11", "r12", "sp", "lr"]
sample_trace_dir = "output_decapsulation_sample_TRIM"
result_dir_sample = "Results_decapsulation_sample"
no_instructions = 500

reset_result_dir = False
if reset_result_dir:
    reset_folder(result_dir_sample)
os.makedirs(result_dir_sample, exist_ok=True)

rebuild_npy = False

if rebuild_npy:
    create_sample_npy_files(result_dir_sample, sample_trace_dir, "HW", sample_registers)
    create_sample_npy_files(result_dir_sample, sample_trace_dir, "HD", sample_registers)


In [3]:
def load_sample_runs(result_dir, leakage_model):
    data_by_run = {}
    fault_order_by_run = {}

    for file in sorted(Path(result_dir).glob(f"{leakage_model}_run_*.npy")):
        match = re.search(r"run_(\d+)$", file.stem)
        if match is None:
            continue

        run_index = int(match.group(1))
        data_by_run[run_index] = np.load(file)

        order_path = Path(result_dir) / f"{leakage_model}_run_{run_index}_fault_order.csv"
        if order_path.exists():
            fault_order_by_run[run_index] = pd.read_csv(order_path)["fault_index"].astype(int).tolist()
        else:
            fault_order_by_run[run_index] = list(range(data_by_run[run_index].shape[0]))

    return data_by_run, fault_order_by_run


data_HW_sample, hw_fault_order = load_sample_runs(result_dir_sample, "HW")
data_HD_sample, hd_fault_order = load_sample_runs(result_dir_sample, "HD")

print("Information about data_HW_sample")
print("--------------------------------")
for run_index, data in sorted(data_HW_sample.items()):
    print(f"Run {run_index}, fault order: {hw_fault_order[run_index]}")
    information(data)

print("\nInformation about data_HD_sample")
print("--------------------------------")
for run_index, data in sorted(data_HD_sample.items()):
    print(f"Run {run_index}, fault order: {hd_fault_order[run_index]}")
    information(data)


Information about data_HW_sample
--------------------------------

Information about data_HD_sample
--------------------------------


In [4]:
run_index = 0
base_fault = 1
faults_to_compare = [2, 4, 8, 16, 32, 64, 128, 256, 512]

if run_index not in data_HW_sample:
    raise ValueError(f"Run {run_index} not found. Available HW runs: {sorted(data_HW_sample)}")

if run_index not in data_HD_sample:
    raise ValueError(f"Run {run_index} not found. Available HD runs: {sorted(data_HD_sample)}")

hw_fault_to_row = {fault: row for row, fault in enumerate(hw_fault_order[run_index])}
hd_fault_to_row = {fault: row for row, fault in enumerate(hd_fault_order[run_index])}

if base_fault not in hw_fault_to_row:
    raise ValueError(f"Base fault {base_fault} not found for run {run_index}. Available faults: {hw_fault_order[run_index]}")

snr_hw = {}
snr_hd = {}

base_hw_row = hw_fault_to_row[base_fault]
base_hd_row = hd_fault_to_row[base_fault]

for alt_fault in faults_to_compare:
    if alt_fault not in hw_fault_to_row or alt_fault not in hd_fault_to_row:
        print(f"Skipping fault {alt_fault}: not present for run {run_index}")
        continue

    alt_hw_row = hw_fault_to_row[alt_fault]
    alt_hd_row = hd_fault_to_row[alt_fault]
    key = f"trace_{run_index}_{base_fault} vs trace_{run_index}_{alt_fault}"

    snr_hw[key] = compute_snr(
        data_HW_sample[run_index][base_hw_row:base_hw_row + 1],
        data_HW_sample[run_index][alt_hw_row:alt_hw_row + 1],
    )
    snr_hd[key] = compute_snr(
        data_HD_sample[run_index][base_hd_row:base_hd_row + 1],
        data_HD_sample[run_index][alt_hd_row:alt_hd_row + 1],
    )

    print_top3_snr_indices(snr_hw[key], f"HW {key}")
    print_top3_snr_indices(snr_hd[key], f"HD {key}")

plot_snr_combined(
    snr_hw,
    f"HW SNR comparison for run {run_index}",
    os.path.join(result_dir_sample, f"snr_HW_run_{run_index}_combined.png"),
    no_instructions=no_instructions,
)

plot_snr_combined(
    snr_hd,
    f"HD SNR comparison for run {run_index}",
    os.path.join(result_dir_sample, f"snr_HD_run_{run_index}_combined.png"),
    no_instructions=no_instructions,
)


ValueError: Run 0 not found. Available HW runs: []